In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)), # TODO: Resize to 28x28
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(), # TODO: Convert to Tensor
    transforms.Normalize(mean=mean, std=std) # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
import glob
from PIL import Image
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns
import pandas as pd
import torchvision.models as models
from sklearn.metrics.pairwise import cosine_similarity
from torchvision.datasets import ImageFolder


In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
train_loader = DataLoader(train_dataset, batch_size= 256, shuffle= True, num_workers= 2, pin_memory= True)
test_loader = DataLoader(test_dataset, batch_size= 256, shuffle= False, num_workers= 2, pin_memory= True)

# Get a batch of training images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")
print("Train loader batch sample X shape (input):", tuple(images.shape), "\nTrain loader batch sample y shape (target):", tuple(labels.shape))

fig, axes = plt.subplots(2, 6, figsize=(12, 4))

for i, ax in enumerate(axes.ravel()):
    x, y = train_dataset[i]

    # Unnormalize
    x_vis = x.clone()
    for c in range(3):
        x_vis[c] = x_vis[c] * std[c] + mean[c]

    x_vis = torch.clamp(x_vis.permute(1, 2, 0), 0, 1)

    ax.imshow(x_vis)
    ax.set_title(f"{i}: {letters[y]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s


# Write your code here
model = efficientnet_v2_s(pretrained= True)
# Modify the classifier
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 26)
#Freeze
model.requires_grad_(False)
model.classifier.requires_grad_(True)
model= model.to(device)



In [ ]:
# Write your code here
def accuracy_from_logits(logits, labels):
    preds = torch.argmax(logits, dim= 1)
    return (preds == labels).float().mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
  model.train()
  total_loss, total_acc = 0.0, 0.0


  for images, labels in loader:
      images = images.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()

      logits = model(images)
      loss = criterion(logits, labels - 1)

      loss.backward()

      optimizer.step()

      total_loss += loss.item()
      total_acc += accuracy_from_logits(logits.detach(), labels)

  return total_loss / len(loader), total_acc / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for images, labels in loader:
          images = images.to(device)
          labels = labels.to(device)

          logits = model(images)

          loss = criterion(logits, labels - 1)

          total_loss += loss.item()
          total_acc += accuracy_from_logits(logits, labels)

    return total_loss / len(loader), total_acc / len(loader)


In [ ]:
# Write your code here
# Training setup

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr= 0.001, weight_decay= 0.0001)

# TO-DO: Set number of epochs
num_epochs = 5  # How many times to iterate through the dataset?

#Scheduler for optimization
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-5
)

# Initialize history tracking
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

# Training loop
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, test_loader, criterion)

    #Scheduler for optimization
    scheduler.step(float(test_acc))
    current_lr = optimizer.param_groups[0]["lr"]

    # Store history
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}, LR: {current_lr:.6f}')


In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), history["train_loss"] , label="train_loss", marker='o')
plt.plot(range(1, num_epochs+1), history["test_loss"], label= "test_loss" , marker='o')
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), history["train_acc"], label="train_acc", marker='o')
plt.plot(range(1, num_epochs+1), history["test_acc"] , label="test_acc", marker='o')
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.show()


In [ ]:
# Write your code here
